# Sprint 2 — Prétraitement & Vectorisation Dataset 1 (EN)

**Auteur** : TAYAR Ali  
**Objectif** : Nettoyer les textes médicaux et les transformer en représentations numériques pour le LSTM.

---



## 0. Imports & Configuration

In [22]:
import pandas as pd
import numpy as np
import nltk
import re
import os
import pickle

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

print('Imports OK')

Imports OK


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\klever\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\klever\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\klever\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\klever\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 1. Chargement du dataset nettoyé (Sprint 1)

In [23]:
df = pd.read_csv('../data/processed/ds1_eda.csv')

print(f'Dataset chargé : {df.shape}')
print(f'Colonnes : {df.columns.tolist()}')
df.head(3)

Dataset chargé : (14438, 5)
Colonnes : ['label', 'text', 'label_name', 'word_count', 'char_count']


,label,text,label_name,word_count,char_count
0,4,Catheterization laboratory events and hospital...,Cardiovascular diseases,234,1686
1,5,Renal abscess in children. Three cases of rena...,General pathological conditions,147,962
2,2,Hyperplastic polyps seen at sigmoidoscopy are ...,Digestive system diseases,142,1031


## 2. Nettoyage du texte

In [24]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

extra_stopwords = {
    'may', 'also', 'one', 'two', 'three', 'four', 'five',
    'less', 'more', 'well', 'using'
}
stop_words = stop_words.union(extra_stopwords)

def clean_text(text):
    # Lowercase
    text = str(text).lower()
    # Supprimer HTML et URLs
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    # Supprimer chiffres isolés
    text = re.sub(r'\b\d+\b', ' ', text)
    # Supprimer ponctuation et caractères spéciaux
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Supprimer espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize_and_lemmatize(text):
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens
              if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

# Appliquer sur tout le dataset
print('Nettoyage en cours...')
df['text_clean'] = df['text'].apply(clean_text)
df['text_processed'] = df['text_clean'].apply(tokenize_and_lemmatize)
print('Nettoyage terminé')

# Comparer avant / après
print('\n--- Exemple avant ---')
print(df['text'].iloc[0][:200])
print('\n--- Exemple après ---')
print(df['text_processed'].iloc[0][:200])

Nettoyage en cours...
Nettoyage terminé

--- Exemple avant ---
Catheterization laboratory events and hospital outcome with direct angioplasty for acute myocardial infarction To assess the safety of direct infarct angioplasty without antecedent thrombolytic therap

--- Exemple après ---
catheterization laboratory event hospital outcome direct angioplasty acute myocardial infarction assess safety direct infarct angioplasty without antecedent thrombolytic therapy catheterization labora


## 3. Vérification post-nettoyage

In [25]:
df['word_count_clean'] = df['text_processed'].str.split().str.len()

print('=== Statistiques après nettoyage ===')
print(df[['word_count', 'word_count_clean']].describe().round(1))
print(f'\nRéduction moyenne : {(1 - df["word_count_clean"].mean() / df["word_count"].mean()) * 100:.1f}%')
print(f'95e percentile après nettoyage : {int(np.percentile(df["word_count_clean"], 95))} mots')

=== Statistiques après nettoyage ===
       word_count  word_count_clean
count     14438.0           14438.0
mean        179.9             104.2
std          76.5              42.4
min          24.0              15.0
25%         122.0              73.0
50%         176.0             102.0
75%         235.0             133.0
max         596.0             374.0

Réduction moyenne : 42.1%
95e percentile après nettoyage : 175 mots


## 4. Encodage des labels

In [26]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

print('=== Encodage des labels ===')
print(f'Classes originales : {le.classes_}')
print(f'Classes encodées   : {sorted(df["label_encoded"].unique())}')
print(f'\nMapping :')
for orig, enc in zip(le.classes_, range(len(le.classes_))):
    print(f'  {orig} → {enc}')

=== Encodage des labels ===
Classes originales : [1 2 3 4 5]
Classes encodées   : [0, 1, 2, 3, 4]

Mapping :
  1 → 0
  2 → 1
  3 → 2
  4 → 3
  5 → 4


## 5. Split Train / Validation / Test

In [27]:
MAXLEN     = 179
VOCAB_SIZE = 10000

X = df['text_processed']
y = df['label_encoded']

# Split 70% train / 15% val / 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train : {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Val   : {len(X_val):,} ({len(X_val)/len(X)*100:.1f}%)')
print(f'Test  : {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)')

Train : 10,106 (70.0%)
Val   : 2,166 (15.0%)
Test  : 2,166 (15.0%)


## 6. Séquences + Padding Keras

In [28]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAXLEN, padding='post', truncating='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAXLEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAXLEN, padding='post', truncating='post')

print(f'Shape X_train_pad : {X_train_pad.shape}')
print(f'Shape X_val_pad   : {X_val_pad.shape}')
print(f'Shape X_test_pad  : {X_test_pad.shape}')
print(f'\nExemple séquence (5 premiers tokens) : {X_train_pad[0][:5]}')

Shape X_train_pad : (10106, 179)
Shape X_val_pad   : (2166, 179)
Shape X_test_pad  : (2166, 179)

Exemple séquence (5 premiers tokens) : [2131  366   48 7710 4920]


## 7. Matrice TF-IDF (pour Feature Selection)

In [29]:
tfidf = TfidfVectorizer(max_features=VOCAB_SIZE)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Shape TF-IDF train : {X_train_tfidf.shape}')
print(f'Shape TF-IDF val   : {X_val_tfidf.shape}')
print(f'Shape TF-IDF test  : {X_test_tfidf.shape}')
print(f'\nTop 10 features TF-IDF :')
features = tfidf.get_feature_names_out()
for f in features[:10]:
    print(f'  {f}')

Shape TF-IDF train : (10106, 10000)
Shape TF-IDF val   : (2166, 10000)
Shape TF-IDF test  : (2166, 10000)

Top 10 features TF-IDF :
  aaa
  aai
  abandoned
  abdomen
  abdominal
  abdominis
  abdominoperineal
  abductor
  aberrant
  aberration


## 8. Sauvegarde des données prétraitées

In [30]:
# Sauvegarder les séquences paddées
np.save(f'{PROCESSED_PATH}X_train_pad.npy', X_train_pad)
np.save(f'{PROCESSED_PATH}X_val_pad.npy',   X_val_pad)
np.save(f'{PROCESSED_PATH}X_test_pad.npy',  X_test_pad)

# Sauvegarder les labels
np.save(f'{PROCESSED_PATH}y_train.npy', y_train.values)
np.save(f'{PROCESSED_PATH}y_val.npy',   y_val.values)
np.save(f'{PROCESSED_PATH}y_test.npy',  y_test.values)

# Sauvegarder TF-IDF et Tokenizer
import pickle
with open(f'{PROCESSED_PATH}tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open(f'{PROCESSED_PATH}keras_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
with open(f'{PROCESSED_PATH}label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# AJOUT — Sauvegarder les textes splittés pour Sprint 4 et 5
X_train.to_csv(f'{PROCESSED_PATH}X_train_text.csv', index=False, header=['text'])
X_val.to_csv(f'{PROCESSED_PATH}X_val_text.csv',     index=False, header=['text'])
X_test.to_csv(f'{PROCESSED_PATH}X_test_text.csv',   index=False, header=['text'])

# Sauvegarder les paramètres
import json
params = {
    'maxlen'     : MAXLEN,
    'vocab_size' : VOCAB_SIZE,
    'n_classes'  : int(y.nunique()),
    'n_train'    : int(len(X_train)),
    'n_val'      : int(len(X_val)),
    'n_test'     : int(len(X_test))
}
with open(f'{PROCESSED_PATH}ds1_params.json', 'w') as f:
    json.dump(params, f, indent=2)

print('Fichiers sauvegardés :')
for f in os.listdir(PROCESSED_PATH):
    size = os.path.getsize(os.path.join(PROCESSED_PATH, f)) / 1024
    print(f'  {f:<35} {size:.1f} KB')

Fichiers sauvegardés :
  ds1_eda.csv                         17878.6 KB
  ds1_params.json                     0.1 KB
  keras_tokenizer.pkl                 1140.2 KB
  label_encoder.pkl                   0.3 KB
  tfidf_vectorizer.pkl                337.1 KB
  X_test_pad.npy                      1514.6 KB
  X_test_text.csv                     1886.3 KB
  X_train_pad.npy                     7066.4 KB
  X_train_text.csv                    8778.6 KB
  X_val_pad.npy                       1514.6 KB
  X_val_text.csv                      1882.6 KB
  y_test.npy                          17.0 KB
  y_train.npy                         79.1 KB
  y_val.npy                           17.0 KB


## 9. Résumé Sprint 2

In [33]:
print('=' * 50)
print('   RÉSUMÉ SPRINT 2 — PRÉTRAITEMENT DS1')
print('=' * 50)
print(f"""
NETTOYAGE
  Lowercase + HTML/URLs supprimés  
  Chiffres isolés supprimés        
  Ponctuation supprimée            
  Stopwords NLTK + extra_stopwords 
    → 'may', 'also', 'one', 'two', 'three',
      'four', 'five', 'less', 'more', 'well', 'using'
  Lemmatisation WordNet            
  Réduction vocabulaire            : 40.6%

SPLIT
  Train : 10,106 (70%)
  Val   : 2,166  (15%)
  Test  : 2,166  (15%)

SÉQUENCES KERAS
  maxlen     : {MAXLEN}
  vocab_size : {VOCAB_SIZE}

TF-IDF
  Shape : (10106, 10000)

FICHIERS SAUVEGARDÉS
  data/processed/ → 14 fichiers
    ├── X_train/val/test_pad.npy
    ├── y_train/val/test.npy
    ├── X_train/val/test_text.csv  ← Sprint 4
    ├── tfidf_vectorizer.pkl
    ├── keras_tokenizer.pkl
    ├── label_encoder.pkl
    └── ds1_params.json

SPRINT 3 COMPLÉTÉ
  Accuracy  : 0.5531
  F1 macro  : 0.5466
""")
print('=' * 50)

   RÉSUMÉ SPRINT 2 — PRÉTRAITEMENT DS1

NETTOYAGE
  Lowercase + HTML/URLs supprimés  
  Chiffres isolés supprimés        
  Ponctuation supprimée            
  Stopwords NLTK + extra_stopwords 
    → 'may', 'also', 'one', 'two', 'three',
      'four', 'five', 'less', 'more', 'well', 'using'
  Lemmatisation WordNet            
  Réduction vocabulaire            : 40.6%

SPLIT
  Train : 10,106 (70%)
  Val   : 2,166  (15%)
  Test  : 2,166  (15%)

SÉQUENCES KERAS
  maxlen     : 179
  vocab_size : 10000

TF-IDF
  Shape : (10106, 10000)

FICHIERS SAUVEGARDÉS
  data/processed/ → 14 fichiers
    ├── X_train/val/test_pad.npy
    ├── y_train/val/test.npy
    ├── X_train/val/test_text.csv  ← Sprint 4
    ├── tfidf_vectorizer.pkl
    ├── keras_tokenizer.pkl
    ├── label_encoder.pkl
    └── ds1_params.json

SPRINT 3 COMPLÉTÉ
  Accuracy  : 0.5531
  F1 macro  : 0.5466

